# Competitive Binding and Concentration Sweeps

This notebook demonstrates how to use equiconc to study competitive
binding and visualize how equilibrium shifts with changing conditions.

In [ ]:
import equiconc
import numpy as np
import matplotlib.pyplot as plt

## Setup: competitive heterodimers

Strand **A** can bind either **B** (forming **AB**) or **C** (forming **AC**).
We fix $[\text{A}]_0 = [\text{C}]_0 = 1\;\mu\text{M}$ and sweep
$[\text{B}]_0$ to see how the competitor displaces **C** from **A**.

- $\Delta G^\circ_{\text{AB}} = -12$ kcal/mol (strong binder)
- $\Delta G^\circ_{\text{AC}} = -10$ kcal/mol (weaker binder)

In [ ]:
conc_A = 1e-6   # 1 uM
conc_C = 1e-6   # 1 uM
dg_AB = -12.0
dg_AC = -10.0

conc_B_range = np.logspace(-8, -4, 80)  # 10 nM to 100 uM

results = {"B0": [], "free_A": [], "AB": [], "AC": [], "free_B": [], "free_C": []}

for conc_B in conc_B_range:
    eq = (
        equiconc.System()
        .monomer("A", conc_A)
        .monomer("B", conc_B)
        .monomer("C", conc_C)
        .complex("AB", [("A", 1), ("B", 1)], delta_g=dg_AB)
        .complex("AC", [("A", 1), ("C", 1)], delta_g=dg_AC)
        .equilibrium()
    )
    results["B0"].append(conc_B)
    results["free_A"].append(eq["A"])
    results["AB"].append(eq["AB"])
    results["AC"].append(eq["AC"])
    results["free_B"].append(eq["B"])
    results["free_C"].append(eq["C"])

## Titration curve

As $[\text{B}]_0$ increases, **AB** formation rises and **AC** is displaced.

In [ ]:
B0_uM = np.array(results["B0"]) * 1e6

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(B0_uM, np.array(results["AB"]) * 1e6, label="[AB]", linewidth=2)
ax.plot(B0_uM, np.array(results["AC"]) * 1e6, label="[AC]", linewidth=2)
ax.plot(B0_uM, np.array(results["free_A"]) * 1e6, label="free [A]",
        linewidth=2, linestyle="--")
ax.set_xscale("log")
ax.set_xlabel("Total [B] (uM)")
ax.set_ylabel("Concentration (uM)")
ax.set_title("Competitive binding: titrating B into A + C")
ax.legend()
ax.set_ylim(bottom=0)
fig.tight_layout()
plt.show()

## Temperature dependence

Next, we fix all concentrations at 1 $\mu$M and sweep temperature
to see how the equilibrium shifts. Higher temperatures favor
dissociation (less negative effective $\Delta G$).

In [ ]:
temperatures_C = np.linspace(10, 90, 80)
temperatures_K = temperatures_C + 273.15

conc_all = 1e-6
dg = -10.0

fraction_bound = []

for T in temperatures_K:
    eq = (
        equiconc.System(temperature_K=T)
        .monomer("A", conc_all)
        .monomer("B", conc_all)
        .complex("AB", [("A", 1), ("B", 1)], delta_g=dg)
        .equilibrium()
    )
    fraction_bound.append(eq["AB"] / conc_all)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(temperatures_C, fraction_bound, linewidth=2, color="teal")
ax.set_xlabel("Temperature (C)")
ax.set_ylabel("Fraction of A bound as AB")
ax.set_title("Melting curve: A + B <=> AB")
ax.set_ylim(-0.02, 1.02)
ax.axhline(0.5, color="gray", linestyle=":", linewidth=1)
fig.tight_layout()
plt.show()

## Affinity comparison

Finally, let's compare how different $\Delta G^\circ$ values affect
the melting curve.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

for dg_val in [-8.0, -10.0, -12.0, -14.0]:
    fractions = []
    for T in temperatures_K:
        eq = (
            equiconc.System(temperature_K=T)
            .monomer("A", conc_all)
            .monomer("B", conc_all)
            .complex("AB", [("A", 1), ("B", 1)], delta_g=dg_val)
            .equilibrium()
        )
        fractions.append(eq["AB"] / conc_all)
    ax.plot(temperatures_C, fractions, linewidth=2,
            label=f"DG = {dg_val} kcal/mol")

ax.set_xlabel("Temperature (C)")
ax.set_ylabel("Fraction bound")
ax.set_title("Melting curves for different binding affinities")
ax.legend()
ax.set_ylim(-0.02, 1.02)
ax.axhline(0.5, color="gray", linestyle=":", linewidth=1)
fig.tight_layout()
plt.show()